In [1]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [2]:
raw = client.query(f"""
    SELECT ingested_at, business_date, payload
    FROM `{PROJECT_ID}.nbu_raw.raw_rates`
""").to_dataframe()

parsed = pd.json_normalize(raw["payload"].map(json.loads))
parsed["ingested_at"] = raw["ingested_at"]
parsed["business_date"] = raw["business_date"]

df = parsed.copy()
df["currency_code"] = df["cc"].str.strip().str.upper()
df["currency_name"] = df["txt"].str.strip()
df["rate"] = df["rate"].astype(float)

print(df.shape)
df.head()

(135, 10)


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cc,exchangedate,r030,rate,special,txt,ingested_at,business_date,currency_code,currency_name
0,DZD,23.09.2026,12,0.33482,NaN,Алжирський динар,2026-09-23 10:52:09.715837+00:00,2026-09-23,DZD,Алжирський динар
1,AUD,23.09.2026,36,31.85160,NaN,Австралійський долар,2026-09-23 10:52:09.715837+00:00,2026-09-23,AUD,Австралійський долар
2,BDT,23.09.2026,50,0.36531,NaN,Така,2026-09-23 10:52:09.715837+00:00,2026-09-23,BDT,Така
3,CAD,23.09.2026,124,31.89250,NaN,Канадський долар,2026-09-23 10:52:09.715837+00:00,2026-09-23,CAD,Канадський долар
4,CNY,23.09.2026,156,6.68500,NaN,Юань Женьміньбі,2026-09-23 10:52:09.715837+00:00,2026-09-23,CNY,Юань Женьміньбі


In [3]:
snap = (df.sort_values(["business_date", "ingested_at"])
           .drop_duplicates(subset="currency_code", keep="last"))

print(snap.shape)
snap[["currency_code", "rate", "business_date", "ingested_at"]].head()

(45, 10)


,currency_code,rate,business_date,ingested_at
90,DZD,0.33625,2026-09-25,2026-09-24 14:11:49.651266+00:00
91,AUD,31.59710,2026-09-25,2026-09-24 14:11:49.651266+00:00
92,BDT,0.36557,2026-09-25,2026-09-24 14:11:49.651266+00:00
93,CAD,31.85520,2026-09-25,2026-09-24 14:11:49.651266+00:00
94,CNY,6.70050,2026-09-25,2026-09-24 14:11:49.651266+00:00


In [4]:
dim_currency = client.query(f"""
    SELECT currency_key, currency_code
    FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
""").to_dataframe()

snap = snap.merge(dim_currency, on="currency_code", how="left")
snap["currency_key"] = snap["currency_key"].fillna(-1).astype(int)

print(snap.shape)
snap[["currency_key", "currency_code", "rate"]].head()

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(45, 11)


,currency_key,currency_code,rate
0,10,DZD,0.33625
1,2,AUD,31.59710
2,4,BDT,0.36557
3,5,CAD,31.85520
4,7,CNY,6.70050


In [5]:
snap["snapshot_ts"] = pd.Timestamp.now(tz="UTC")

snap_final = snap[["currency_key", "currency_code", "currency_name",
                    "rate", "business_date", "snapshot_ts"]]
snap_final.head()

,currency_key,currency_code,currency_name,rate,business_date,snapshot_ts
0,10,DZD,Алжирський динар,0.33625,2026-09-25,2026-09-25 08:33:17.060960+00:00
1,2,AUD,Австралійський долар,31.59710,2026-09-25,2026-09-25 08:33:17.060960+00:00
2,4,BDT,Така,0.36557,2026-09-25,2026-09-25 08:33:17.060960+00:00
3,5,CAD,Канадський долар,31.85520,2026-09-25,2026-09-25 08:33:17.060960+00:00
4,7,CNY,Юань Женьміньбі,6.70050,2026-09-25,2026-09-25 08:33:17.060960+00:00


In [6]:
snap_schema = [
    bigquery.SchemaField("currency_key",   "INTEGER"),
    bigquery.SchemaField("currency_code",  "STRING"),
    bigquery.SchemaField("currency_name",  "STRING"),
    bigquery.SchemaField("rate",           "FLOAT"),
    bigquery.SchemaField("business_date",  "DATE"),
    bigquery.SchemaField("snapshot_ts",    "TIMESTAMP"),
]

SNAP_TABLE_ID = f"{PROJECT_ID}.nbu_dwh.snapshot_rates_current"

cfg = bigquery.LoadJobConfig(schema=snap_schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(snap_final, SNAP_TABLE_ID, job_config=cfg).result()

print("записано рядків:", len(snap_final))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано рядків: 45


In [8]:
check = client.query(f"""
    SELECT currency_key, currency_code, currency_name, r030
    FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
    ORDER BY currency_code
    LIMIT 10
""").to_dataframe()

check

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,currency_key,currency_code,currency_name,r030
0,1,AED,Дирхам ОАЕ,784
1,2,AUD,Австралійський долар,36
2,3,AZN,Азербайджанський манат,944
3,4,BDT,Така,50
4,5,CAD,Канадський долар,124
5,6,CHF,Швейцарський франк,756
6,7,CNY,Юань Женьміньбі,156
7,8,CZK,Чеська крона,203
8,9,DKK,Данська крона,208
9,10,DZD,Алжирський динар,12


In [11]:
# я прибрала currency_key з попередніх запусків бо був повторний merge, який створює currency_key_x/_y і видавало KeyError. я пару разів перезапускала, шось з клодом пробувала, але не вийшло і довелось до такого варіанту прийти

for col in ["currency_key", "currency_key_x", "currency_key_y"]:
    if col in snap.columns:
        snap = snap.drop(columns=col)

dim_currency = client.query(f"""
    SELECT currency_key, currency_code
    FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
""").to_dataframe()

snap = snap.merge(dim_currency, on="currency_code", how="left")
snap["currency_key"] = snap["currency_key"].fillna(-1).astype(int)

print(snap.shape)
snap[["currency_key", "currency_code", "rate"]].head()

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(45, 12)


,currency_key,currency_code,rate
0,10,DZD,0.33625
1,2,AUD,31.59710
2,4,BDT,0.36557
3,5,CAD,31.85520
4,7,CNY,6.70050


In [12]:
dim_currency = client.query(f"""
    SELECT currency_key, currency_code
    FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
""").to_dataframe()

print("Колонки dim_currency:", list(dim_currency.columns))
print()

merged = snap.merge(dim_currency, on="currency_code", how="left")
print("Колонки ПІСЛЯ merge:", list(merged.columns))

Колонки snap ДО merge: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt', 'ingested_at', 'business_date', 'currency_code', 'currency_name', 'snapshot_ts', 'currency_key']



/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Колонки dim_currency: ['currency_key', 'currency_code']

Колонки ПІСЛЯ merge: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt', 'ingested_at', 'business_date', 'currency_code', 'currency_name', 'snapshot_ts', 'currency_key_x', 'currency_key_y']


In [13]:
snap["snapshot_ts"] = pd.Timestamp.now(tz="UTC")

snap_final = snap[["currency_key", "currency_code", "currency_name",
                   "rate", "business_date", "snapshot_ts"]]

print(snap_final.shape)
snap_final.head()

(45, 6)


,currency_key,currency_code,currency_name,rate,business_date,snapshot_ts
0,10,DZD,Алжирський динар,0.33625,2026-09-25,2026-09-28 11:24:08.378300+00:00
1,2,AUD,Австралійський долар,31.59710,2026-09-25,2026-09-28 11:24:08.378300+00:00
2,4,BDT,Така,0.36557,2026-09-25,2026-09-28 11:24:08.378300+00:00
3,5,CAD,Канадський долар,31.85520,2026-09-25,2026-09-28 11:24:08.378300+00:00
4,7,CNY,Юань Женьміньбі,6.70050,2026-09-25,2026-09-28 11:24:08.378300+00:00


In [14]:
snap_schema = [
    bigquery.SchemaField("currency_key",   "INTEGER"),
    bigquery.SchemaField("currency_code",  "STRING"),
    bigquery.SchemaField("currency_name",  "STRING"),
    bigquery.SchemaField("rate",           "FLOAT"),
    bigquery.SchemaField("business_date",  "DATE"),
    bigquery.SchemaField("snapshot_ts",    "TIMESTAMP"),
]

SNAP_TABLE_ID = f"{PROJECT_ID}.nbu_dwh.snapshot_rates_current"

# WRITE_TRUNCATE: снепшот показує тільки "зараз", тому щоразу перезаписуємо таблицю
cfg = bigquery.LoadJobConfig(schema=snap_schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(snap_final, SNAP_TABLE_ID, job_config=cfg).result()

print("записано рядків:", len(snap_final))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано рядків: 45


In [15]:
check_sql = f"""
SELECT
  COUNT(*)                      AS total_rows,
  COUNT(DISTINCT currency_code) AS uniq_codes,
  COUNTIF(currency_key = -1)    AS unmatched
FROM `{PROJECT_ID}.nbu_dwh.snapshot_rates_current`
"""
print(client.query(check_sql).to_dataframe())

bronze_sql = f"""
SELECT COUNT(DISTINCT JSON_VALUE(payload, '$.cc')) AS bronze_currencies
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""
print(client.query(bronze_sql).to_dataframe())

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   total_rows  uniq_codes  unmatched
0          45          45          0
   bronze_currencies
0                 45


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
